# ENDPOINT MATCH

Ensuring that the columns from crawl 2 match the legacy DataFrames.

In [17]:
# ==== IMPORTS ====
import pandas as pd
import json


# ==== SETTINGS ====
# ---- IMPORT SETTINGS ----
IMPORT_PATH_EXTERNAL = "/Users/oliver/Desktop/MSc_Speciale/ThesisDataRepo/data/gcp_manual_copy/"
IMPORT_PATH_INTERNAL = "../Data/"

# ---- EXPORT SETTINGS ----
EXPORT = True
EXPORT_PATH = IMPORT_PATH_INTERNAL

# ---- FILES ----
# Metrics files
FILE_EXTERNAL = "thesis_meta_all_metrics_except_grade_27032026.parquet"
FILE_INTERNAL = "extracted_metrics_unified_test2_enriched.csv"

# ==== FUNCTION ====
def load_csv_to_df(csv_path, sep=";", print=True):
    try:
        df = pd.read_csv(csv_path, encoding="utf-8", sep=sep)
        if print:
            print(f"Successfully loaded CSV from {csv_path}")
            print(f"DataFrame shape: {df.shape}")
            print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading CSV from {csv_path}: {e}")
        return None

def load_parquet_to_df(parquet_path, na=False, print=True):
    try:
        df = pd.read_parquet(parquet_path)
        if print:
            print(f"Successfully loaded Parquet from {parquet_path}")
            print(f"DataFrame shape: {df.shape}")
        if na:
            if print:
                print(f"DataFrame N/A counts:\n{df.isna().sum()}\n")
        if print:
            print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading Parquet from {parquet_path}: {e}")
        return None



In [22]:
# ==== LOAD DATAFRAMES ====
df_legacy = load_parquet_to_df(IMPORT_PATH_EXTERNAL + FILE_EXTERNAL, print=False)
df_crawl2 = load_csv_to_df(IMPORT_PATH_INTERNAL + FILE_INTERNAL, sep=",", print=False)

# ==== COLUMNS TO DROP ====
drop_columns = [
    "access_ss",
    "Affiliations",
    "collection_facet",
    "format",
    "fulltext_availability_facet",
    "ISBN",
    "Journal Page",
    "isolanguage_facet",
    "Publisher",
    "Source",
    "source_all_ss",
    "match_trigger",
    "equation_pipeline_version",
    "pdf_file_analysis",
    "num_tot_pages_analysis",
    "num_cont_pages_analysis",
    "num_words_full_analysis",
    "num_words_cont_analysis",
    "abstract_ts_analysis",
    "Author_analysis",
    "Publication Year_analysis",
    "primary_member_id_s_analysis",
    "Title_analysis",
    "department_match_fragment",
    "department_match_source",
    "department_match_score",
    "linguistics_backend",
    "department_match_alias",
    "corrupt_cid",
    ]

# ==== DROP COLUMNS ====
df_legacy = df_legacy.drop(columns=drop_columns, errors="ignore")
df_crawl2 = df_crawl2.drop(columns=drop_columns, errors="ignore")

# ==== DISPLAY INFO ====
print("Legacy DataFrame Info:")
print(f"Number of columns: {len(df_legacy.columns)}")
print(df_legacy.columns)
print("\nCrawl2 DataFrame Info:")
print(f"Number of columns: {len(df_crawl2.columns)}")
print(df_crawl2.columns)

Legacy DataFrame Info:
Number of columns: 53
Index(['abstract_ts', 'Timestamp', 'Author', 'citation_count_i', 'ID',
       'dtu_library_collection_facet', 'Publication Year', 'Conference', 'DOI',
       'Editor', 'embargo_ssf', 'has_openaccess_fulltext_b', 'holdings_ssf',
       'Journal Issue', 'journal_issue_tsort', 'journal_oa_model_ss',
       'journal_page_start_tsort', 'Journal Title', 'journal_title_facet',
       'toc_key_s', 'Journal Volume', 'journal_vol_tsort', 'keywords_ts',
       'keywords_facet', 'keywords_normalized', 'member_id_ss', 'ORCID',
       'primary_member_id_s', 'Title', 'MASTER THESIS TITLE', 'BY',
       'SUPERVISED BY', 'YEAR', 'PUBLISHER', 'TYPES', 'pdf_file',
       'num_tot_pages', 'num_cont_pages', 'num_words_full', 'num_words_cont',
       'handin_month', 'num_figures', 'num_tables', 'num_references',
       'equation_count', 'pdf_sha256', 'total_sentences', 'total_words',
       'unique_words', 'avg_sentence_length', 'avg_word_length',
       'lexical

In [19]:
# seeing what columns are in one but not the other
legacy_columns = set(df_legacy.columns)
crawl2_columns = set(df_crawl2.columns)
columns_only_in_legacy = legacy_columns - crawl2_columns
columns_only_in_crawl2 = crawl2_columns - legacy_columns
print(f"Columns only in legacy DataFrame: {columns_only_in_legacy}")
print(f"Columns only in crawl2 DataFrame: {columns_only_in_crawl2}")

Columns only in legacy DataFrame: {'Journal Volume', 'equation_count', 'TYPES', 'Journal Issue', 'journal_title_facet', 'journal_issue_tsort', 'Journal Title', 'SUPERVISED BY', 'PUBLISHER', 'has_openaccess_fulltext_b', 'keywords_normalized', 'BY', 'keywords_facet', 'toc_key_s', 'embargo_ssf', 'member_id_ss', 'journal_page_start_tsort', 'journal_oa_model_ss', 'DOI', 'ID', 'pdf_sha256', 'MASTER THESIS TITLE', 'keywords_ts', 'dtu_library_collection_facet', 'YEAR', 'Editor', 'pdf_file', 'ORCID', 'holdings_ssf', 'Timestamp', 'journal_vol_tsort', 'Conference', 'citation_count_i'}
Columns only in crawl2 DataFrame: {'flesch_kincaid_grade', 'filename', 'author_count'}


In [24]:
# Renaming columns in crawl2 to match legacy for merging
df_crawl2 = df_crawl2.rename(columns={
    "Title": "MASTER THESIS TITLE",
    "Author": "BY",
})

In [26]:
print(df_crawl2.columns.tolist())

['filename', 'num_tot_pages', 'num_cont_pages', 'num_words_full', 'num_words_cont', 'num_figures', 'num_tables', 'num_references', 'total_sentences', 'total_words', 'unique_words', 'avg_sentence_length', 'avg_word_length', 'lexical_diversity', 'flesch_kincaid_grade', 'handin_month', 'primary_member_id_s', 'abstract_ts', 'BY', 'Publication Year', 'MASTER THESIS TITLE', 'Department_new', 'author_count']


- [x] **`Publication Year`**: The year of publication
- [x] **`MASTER THESIS TITLE`**: The english title of the thesis
- [x] **`BY`**: The author(s) in the format "lastname, name" (if multiple auhtors, they're separated wiht ";") 
- [] **`SUPERVISED BY`**: The supervisor(s) in the format "lastname, name" (if multiple supervisors, they're separated with ";")
- [x] **`num_tot_pages`**: Number of total pages in .pdf file
- [x] **`num_cont_pages`**: Number of content pages in the .pdf file (excluding appendix, references etc.)
- [x] **`handin_month`**: The month of handin exstracted from the .pdf file. *OBS(!):* disregard the year in the stirng, and use the metric `Publication Year` for true year.
- [x] **`num_figures`**: Number of figures in the .pdf file
- [x] **`num_tables`**: Number of tables in the .pdf file
- [x] **`num_references`** Number of references listed in the section regarding bibliography in the .pdf file
- [] **`equation_count`**: Number of equations in the .pdf file
- [x] **`total_sentences`**: Number of sentences in main content of .pdf file
- [x] **`total_words`**: Number of words in main content of .pdf file
- [x] **`unique_words`**: Number of unique words in main content of .pdf file
- [x] **`avg_sentence_length`**: Average sentence lenght of main content of .pdf file
- [x] **`avg_word_length`**: Average word lenght of main conent of .pdf file
- [x] **`lexical_diversity`**: Measure of the lexical diversity in the main content of .pdf file (unique_words/total_words)
- [x] **`flesch_kincaid_grade`**: ...
- [x] **`Department_new`**: The department of DTU from which the thesis is published
- [x] **`num_authors`**: Number of authors for MSc Thesis, count of semicolons inn column `BY`. 
If the value is missing (NaN), fillna(0) treats it as 0 semicolons, resulting in 1 author.
- [] **`handin_month_num`** getting only the month from column `handin_month` and mapping to a number (1-12) using the calendar module for robustness.


**WILL BE APPENDED LATER:**
- [] **`grading_scientific_contribution`**: Sub grading score, (x-y)
- [] **`grading_methodological_rigor`**: Sub grading score, (x-y)
- [] **`grading_technical_implementation`**: Sub grading score, (x-y)
- [] **`grading_literature_review`**: Sub grading score, (x-y)
- [] **`grading_process_professionalism`**: Sub grading score, (x-y)
- [] **`grading_impact_applicability`**: Sub grading score, (x-y)
- [] **`grading_research_question_alignment`**: Sub grading score, (x-y)
- [] **`grading_total_score`**: Total assigned grading score (1-100) for the thesis by local LLM. Consistes of the scores; scientific contribution, methodological rigor, technical implementation, literature review, process professionalism, impact applicability.